## Statistical Test: Independent Two-Sample T-Test

To test whether FIFA ranking advantage is related to match outcomes, we compare the average `rank_difference` for matches won by the home team versus matches won by the away team.

We define:

`rank_difference = away_rank - home_rank`

Because lower FIFA rank numbers are better, a positive `rank_difference` means the home team was ranked better than the away team.

## Imports and Load Data

In [1]:
import pandas as pd
from scipy.stats import ttest_ind

# Load final feature-engineered dataset
df = pd.read_csv("../data/processed/match_data.csv")

df.head()

,date,home_team,away_team,home_score,away_score,tournament,neutral,home_recent_win_rate,away_recent_win_rate,home_recent_goal_diff,away_recent_goal_diff,home_rank,away_rank,rank_difference,result
0,1993-01-20,Zambia,Namibia,4,0,FIFA World Cup qualification,False,1.0,0.0,2.0,-1.0,32.0,158.0,126.0,home_win
1,1993-01-31,Tunisia,Ethiopia,3,0,FIFA World Cup qualification,False,1.0,0.0,5.0,-1.0,38.0,85.0,47.0,home_win
2,1993-01-31,Zimbabwe,Angola,2,1,FIFA World Cup qualification,False,0.5,0.0,0.5,0.0,54.0,102.0,48.0,home_win
3,1993-01-31,Morocco,Benin,5,0,FIFA World Cup qualification,False,1.0,0.0,1.0,-5.0,41.0,127.0,86.0,home_win
4,1993-01-31,Egypt,Togo,3,0,FIFA World Cup qualification,False,0.0,0.0,0.0,-1.0,21.0,101.0,80.0,home_win


## Hypotheses

**Null Hypothesis (H₀):**  
The average `rank_difference` is the same for matches won by the home team and matches won by the away team.

**Alternative Hypothesis (H₁):**  
The average `rank_difference` is different for matches won by the home team and matches won by the away team.

## Prepare Data for Test

In [3]:
# Keep only matches with a winner
# Draws are excluded because we are comparing home wins and away wins
test_df = df[df["result"].isin(["home_win", "away_win"])].copy()

# Split rank_difference by match result
home_win_rank_diff = test_df[test_df["result"] == "home_win"]["rank_difference"]
away_win_rank_diff = test_df[test_df["result"] == "away_win"]["rank_difference"]

print("Number of home wins:", len(home_win_rank_diff))
print("Number of away wins:", len(away_win_rank_diff))

Number of home wins: 3838
Number of away wins: 2239


## Summary Statistics

In [4]:
print("Mean rank_difference for home wins:", round(home_win_rank_diff.mean(), 4))
print("Mean rank_difference for away wins:", round(away_win_rank_diff.mean(), 4))

print("\nMedian rank_difference for home wins:", round(home_win_rank_diff.median(), 4))
print("Median rank_difference for away wins:", round(away_win_rank_diff.median(), 4))

print("\nStandard deviation for home wins:", round(home_win_rank_diff.std(), 4))
print("Standard deviation for away wins:", round(away_win_rank_diff.std(), 4))

Mean rank_difference for home wins: 30.6443
Mean rank_difference for away wins: -38.8941

Median rank_difference for home wins: 28.0
Median rank_difference for away wins: -34.0

Standard deviation for home wins: 48.8531
Standard deviation for away wins: 49.1361


## Run Independent Two-Sample T-Test

In [7]:
# Run independent two-sample t-test
# equal_var=False uses Welch's t-test, which is safer when group variances may differ
t_stat, p_value = ttest_ind(
    home_win_rank_diff,
    away_win_rank_diff,
    equal_var=False
)

print("Statistical Test: Independent two-sample t-test")
print("Question: Is average rank_difference different between home wins and away wins?")

print("\nT-statistic:", round(t_stat, 4))
if p_value < 0.001:
    print("P-value: < 0.001")
else:
    print("P-value:", round(p_value, 4))

Statistical Test: Independent two-sample t-test
Question: Is average rank_difference different between home wins and away wins?

T-statistic: 53.3311
P-value: < 0.001


## Interpret Test Result

In [6]:
alpha = 0.05

if p_value < alpha:
    print("Conclusion: Reject the null hypothesis.")
    print(
        "There is statistically significant evidence that the average rank_difference "
        "differs between matches won by the home team and matches won by the away team."
    )
else:
    print("Conclusion: Fail to reject the null hypothesis.")
    print(
        "There is not enough statistically significant evidence to conclude that "
        "average rank_difference differs between home wins and away wins."
    )

Conclusion: Reject the null hypothesis.
There is statistically significant evidence that the average rank_difference differs between matches won by the home team and matches won by the away team.


### Statistical Test Result

We performed an independent two-sample t-test to compare the average `rank_difference` between matches won by the home team and matches won by the away team.

The null hypothesis was that the average `rank_difference` is the same for home wins and away wins. The alternative hypothesis was that the average `rank_difference` is different between home wins and away wins.

For home wins, the mean `rank_difference` was 30.64. For away wins, the mean `rank_difference` was -38.89. Since `rank_difference = away_rank - home_rank`, positive values mean the home team was ranked better, while negative values mean the away team was ranked better.

The independent two-sample t-test produced a t-statistic of 53.33 and a p-value less than 0.001. Since the p-value is below 0.05, we reject the null hypothesis.

This suggests that ranking advantage is significantly related to match outcome. When the home team wins, the home team tends to be ranked better than the away team. When the away team wins, the away team tends to be ranked better than the home team. This supports using FIFA ranking-based features in our predictive model.

However, this result should not be interpreted causally. Rankings do not directly cause teams to win. Instead, rankings summarize past performance and are also related to factors such as team quality, recent form, opponent strength, and tournament context.